# Splice DNA — tập phổ biến quanh ranh giới

Một chuỗi DNA được viết bằng bốn base A, C, G, T. Trong bộ dữ liệu này, exon là đoạn được giữ trong RNA trưởng thành, intron là đoạn được loại trong quá trình nối RNA. Điểm nối có hai phía. **EI** là exon → intron, **IE** là intron → exon và **N** là mẫu không thuộc hai lớp điểm nối. Ta hỏi tổ hợp base **tại những vị trí nào** xuất hiện thường xuyên trong từng lớp.

| Thành phần | Dùng để làm gì |
|---|---|
| Python chuẩn (`csv`, `zipfile`, `json`, `hashlib`, `tomllib`), curl | Tải bản học thuật, đọc ZIP UCI dự phòng, kiểm SHA-256, đọc records và kết quả |
| Julia Base và thư viện chuẩn `TOML`, `SHA` | Chuẩn hóa input, mã hóa vị trí–base, ECLAT từng lớp và đếm lại |
| IPython và Matplotlib | Bảng audit, bảng support và biểu đồ so sánh theo lớp |
| Java và SPMF 2.64b | Oracle độc lập cho từng map đã khai thác |

Dùng bộ **Molecular Biology (Splice-junction Gene Sequences)** của UCI. Nguồn dự phòng là thành viên `splice.data` trong ZIP UCI, được chấp nhận khi có cùng SHA-256 với bản Szeged. Ghi nguồn UCI và điều kiện CC BY 4.0. [UCI và điều kiện sử dụng](https://doi.org/10.24432/C5M888) · [Nguồn và checksum](../data/manifest.csv)

## Môi trường và thư mục kết quả

Dùng Colab CPU hoặc Jupyter từ clone đầy đủ. Chọn **Run all** từ đầu. Ô khởi tạo in commit, phiên bản môi trường và tạo thư mục kết quả riêng. Ở Colab nó clone public `main`. Trên máy cá nhân cần Julia ≥1.9, Python ≥3.11, Java ≥21 và curl, theo [README](../README.md).

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path(tempfile.mkdtemp(prefix="eclat-", dir="/content"))
    URL = "https://github.com/phuctran33/-DM-Eclat.git"
    REF = "main"
    subprocess.run(["git", "clone", "--single-branch", "--branch", REF, URL, str(ROOT)], check=True)
else:
    ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "src/EclatProject.jl").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Open the notebook from the project checkout.")

sys.path.insert(0, str(ROOT / "notebooks"))
from eclat_setup import csv_rows, run, run_julia, setup, show_table

OUTPUT, JULIA, JAVA = setup(ROOT, IN_COLAB, "02_splice_dna")

## 1. Dữ liệu quan sát

Một record gồm nhãn, source ID và 60 ký hiệu DNA. Các lớp không đều kích thước. Vì vậy, tần suất mẫu cần được chia cho số record giữ lại **trong lớp đó**. Input có mã IUPAC mơ hồ. Bản Julia loại record nếu có ký hiệu ngoài A/C/G/T ở cửa sổ dùng để khai thác, giữ record có ký hiệu mơ hồ chỉ ở ngoài cửa sổ. Sai cấu trúc record bị báo lỗi. Ta in cửa sổ quanh tâm của vài record ngay từ nguồn.

In [ ]:
from shutil import copyfile
import csv
import json
import tomllib
from collections import Counter

run(sys.executable, "-B", "scripts/prepare_application_data.py", "--dataset", "splice")
source_record = OUTPUT / "dataset-source.json"
copyfile(ROOT / "data/applications/splice_source.json", source_record)
print("Nguồn tải và SHA-256:", source_record)
with (ROOT / "data/applications/splice.data").open(encoding="utf-8", newline="") as stream:
    records = [(label.strip(), source.strip(), "".join(sequence.split()).upper())
               for label, source, sequence in csv.reader(stream)]
classes = ("EI", "IE", "N")
samples = [next(row for row in records if row[0] == label) for label in classes]
show_table(["Nhãn", "Source ID", "Độ dài", "Cửa sổ hai phía", "Ký hiệu mơ hồ trong cửa sổ"],
           [(label, source, len(sequence), sequence[24:30] + " | " + sequence[30:36],
             ", ".join(sorted(set(sequence[24:36]) - set("ACGT"))) or "0")
            for label, source, sequence in samples])
raw_counts = Counter(label for label, _, _ in records)
print("Tổng records:", len(records), "· theo lớp:", dict(raw_counts), "· độ dài:", sorted({len(r[2]) for r in records}))

## 2. Mã hóa position–base thành giao dịch

Trong indexing Julia, cửa sổ 25:36 lấy sáu vị trí hai phía ranh giới 30|31. Đây là ranh giới được gán nhãn ở EI/IE và tâm so sánh ở N.

```text
vị trí trong chuỗi   25  26  27  28  29  30  |  31  32  33  34  35  36
vị trí tương đối     -6  -5  -4  -3  -2  -1  |  +1  +2  +3  +4  +5  +6
```

Nếu chỉ gán bốn ID cho A/C/G/T thì các vị trí chứa cùng một base bị nhập làm một. Ở đây ID bằng $4(s-1)+b$, với $s$ là thứ tự vị trí từ 1 đến 12, $b=1,2,3,4$ lần lượt cho A,C,G,T. Một quan sát hợp lệ sinh 12 item, một cho mỗi vị trí. Bảng có 48 ID có thể dùng. Nhãn để chia giao dịch, được lưu cùng source ID ngoài tập item.

## 3. Chạy ECLAT riêng mỗi lớp

Minsup 20% yêu cầu mẫu lặp lại tương đối thường xuyên trong lớp. Với $n_c$ mẫu giữ ở lớp $c$, ngưỡng là $\lceil0{,}2n_c\rceil$. Julia chạy hai backend trên từng lớp và so đủ map. Java SPMF kiểm lại ba map trên cùng input theo từng lớp.

In [ ]:
app = OUTPUT / "applications"
run_julia("scripts/run_applications.jl", "--dataset", "splice", "--output", app)
run(sys.executable, "-B", "scripts/verify_application_results.py", "--dataset", "splice",
    "--output", app, "--java", JAVA)

prepared = app / "prepared/splice"
observations = csv_rows(prepared / "observations.csv")
audit = {(r["class"], r["reason"]): int(r["count"]) for r in csv_rows(prepared / "preparation_audit.csv")}
stats = tomllib.loads((app / "run.toml").read_text(encoding="utf-8"))["statistics"]["dna"]
show_table(["Lớp", "Raw", "Giữ", "Loại vì mơ hồ", "Min count", "Tập phổ biến"], [
    (label, raw_counts[label], audit[label, "kept"], audit[label, "ambiguous_window"],
     stats["classes"][label]["min_count"], stats["classes"][label]["n_patterns"])
    for label in classes
])
print("Tổng giữ:", stats["n_transactions"], "· số ID vị trí–base:", stats["n_items"])
labels = {int(r["item_id"]): (int(r["position"]), r["base"]) for r in csv_rows(prepared / "items.csv")}
numeric = (prepared / "transactions.txt").read_text(encoding="utf-8").splitlines()
for i in [next(i for i, r in enumerate(observations) if r["class"] == label) for label in classes]:
    row = observations[i]
    source_window = next(seq[24:36] for label, source, seq in records if (label, source) == (row["class"], row["source_id"]))
    items = list(map(int, numeric[i].split()))
    decoded = "".join(labels[item][1] for item in items)
    assert source_window == decoded
    show_table(["Lớp", "Source ID", "Cửa sổ", "Giao dịch số"], [
        (row["class"], row["source_id"], decoded[:6] + " | " + decoded[6:], numeric[i])])

## 4. Đếm lại ứng viên và so sánh support

Lấy hợp các tập frequent từ ba lớp, sau đó giữ độ dài 2–4 cho bảng so sánh. Kết quả mining đầy đủ vẫn chứa các độ dài khác. Với mỗi ứng viên, đếm trực tiếp trên giao dịch của cả ba lớp, kể cả lớp mà ứng viên không đạt 20%. Đây là cách giữ các count dương dưới ngưỡng. Mẫu số của từng lớp là số record giữ của lớp đó.

Xếp hạng theo độ chênh $\max_c s_c-\min_c s_c$ giảm dần, sau đó support lớn nhất giảm dần, độ dài và ID tăng dần. $s_c$ là số record lớp $c$ chứa ứng viên chia cho $n_c$. Chênh lệch là điểm phần trăm.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

rows = csv_rows(app / "dna_patterns.csv")
top = rows[:5]
example4 = next(row for row in rows if int(row["length"]) == 4)
examples = [*top, example4] if example4 not in top else top
show_table(["Hạng", "Vị trí–base", "EI count / %", "IE count / %", "N count / %", "Chênh lệch (đpt)"], [
    (row["rank"], row["pattern_labels"].replace(" | ", ", "),
     *(f'{row[f"count_{label}"]} / {100 * float(row[f"support_{label}"]):.2f}%' for label in classes),
     f'{100 * float(row["support_span"]):.2f}')
    for row in examples
])

first_two = [next(row for row in rows if row["pattern_ids"] == ids) for ids in ("27 32", "17 23")]
fig, ax = plt.subplots(figsize=(6.7, 3.5), layout="constrained")
for i, (label, color) in enumerate(zip(classes, ("#31708e", "#ad6235", "#657c70"))):
    ax.bar([j + 0.25 * (i - 1) for j in range(2)],
           [100 * float(row[f"support_{label}"]) for row in first_two], width=0.24, color=color, label=label)
ax.set_xticks([0, 1], ["GT (+1, +2)", "AG (−2, −1)"])
ax.set(title="Support của hai cặp base theo lớp", ylabel="Support trong lớp (%)", ylim=(0, 110))
ax.legend()
chart = OUTPUT / "splice-support.png"
fig.savefig(chart, dpi=140)
plt.close(fig)
display(Image(filename=str(chart)))
validation = json.loads((app / "validation.json").read_text(encoding="utf-8"))
print("Ứng viên:", validation["n_dna_candidates"], "· cặp ứng viên/lớp dưới ngưỡng nhưng count dương:",
      validation["nonzero_infrequent_class_counts"])
print("Bằng chứng:", app / "validation.json")

## 5. Diễn giải

Cặp GT ở hai vị trí ngay sau ranh giới xuất hiện trong 758/765 mẫu EI, khoảng 99,08%. Mẫu vẫn có 98/1.654 lần ở N, khoảng 5,93%. Cặp AG ngay trước ranh giới đạt 762/766 ở IE, khoảng 99,48%, trong khi ở EI là 388/765. Điều này phù hợp với hai dinucleotide thường gặp ở vị trí cho và nhận của intron ở động vật có vú. Mẫu 4 item trong bảng đặt thêm điều kiện nên support EI thấp hơn một số mẫu hai item.

Đây là kết quả mô tả từ dữ liệu đã căn chỉnh. Cửa sổ, loại record mơ hồ và ngưỡng 20% tạo ra tập mẫu nhìn thấy. Lớp N lớn hơn và cũng có thể chứa GT hoặc AG quanh tâm đối chứng. Muốn suy luận khả năng nhận diện điểm nối trên dữ liệu mới cần đánh giá độc lập. [Nghiên cứu Burset và cộng sự](https://doi.org/10.1093/nar/28.21.4364) · [CSV kết quả công bố](../results/applications/dna_patterns.csv)